
# TP — k-NN : Classification sur une base de données *diabetes*

**Module : Intelligence Artificielle / Analyse de données**  
**Auteur : Mouna Mkacher**  
**Classe : EPI DS**

## Objectifs du TP

À la fin de ce TP, vous serez capables de :

1. comprendre le principe de l’algorithme **k-Nearest Neighbors** ou **k-NN** ;
2. charger et explorer une base de données ;
3. préparer les données pour un modèle de Machine Learning ;
4. entraîner un classifieur k-NN ;
5. évaluer le modèle avec l’accuracy, la matrice de confusion et le rapport de classification ;
6. choisir une bonne valeur de `k` ;
7. comprendre pourquoi la **normalisation** des variables est importante pour k-NN.



## 1. Rappel théorique : principe de k-NN

L’algorithme **k-NN** est une méthode de classification supervisée.

Pour classer un nouvel individu, k-NN suit les étapes suivantes :

1. il calcule la distance entre le nouvel individu et tous les exemples de la base d’entraînement ;
2. il sélectionne les **k voisins les plus proches** ;
3. il attribue au nouvel individu la classe majoritaire parmi ces voisins.

Exemple : si `k = 5` et que 3 voisins appartiennent à la classe 1 alors que 2 appartiennent à la classe 0, le nouvel individu est classé en **classe 1**.

La distance la plus utilisée est la distance euclidienne :

\[
d(x, y) = \sqrt{\sum_{i=1}^{n}(x_i-y_i)^2}
\]

Comme k-NN dépend directement des distances, les variables doivent être mises à la même échelle.


In [ ]:

# Installation éventuelle si besoin :
# !pip install pandas numpy matplotlib scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)



## 2. Chargement de la base de données

Nous allons utiliser la base **Diabetes** disponible dans `scikit-learn`.

Attention : dans `scikit-learn`, la variable cible originale est une valeur continue qui mesure la progression de la maladie. Pour faire un TP de **classification**, nous allons transformer cette cible en deux classes :

- `0` : progression faible ou moyenne ;
- `1` : progression élevée.

La séparation est faite à partir de la médiane de la variable cible.


In [ ]:

# Charger la base de données diabetes
diabetes = load_diabetes()

# Convertir en DataFrame
X = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y_regression = pd.Series(diabetes.target, name="progression")

# Transformer le problème en classification binaire
# Classe 1 si la progression est supérieure à la médiane, sinon classe 0
threshold = y_regression.median()
y = (y_regression > threshold).astype(int)
y.name = "risque_eleve"

# Fusionner les données pour l'exploration
df = X.copy()
df["progression"] = y_regression
df["risque_eleve"] = y

df.head()



## 3. Exploration rapide des données

Les variables de la base sont déjà codées numériquement. On dispose notamment de variables relatives à l’âge, au sexe, à l’indice de masse corporelle et à des mesures sanguines anonymisées.


In [ ]:

# Dimensions de la base
print("Nombre de lignes :", df.shape[0])
print("Nombre de colonnes :", df.shape[1])

# Informations générales
df.info()


In [ ]:

# Statistiques descriptives
df.describe()


In [ ]:

# Répartition des classes
class_counts = y.value_counts().sort_index()
class_counts.index = ["0 = risque faible/moyen", "1 = risque élevé"]
class_counts


In [ ]:

# Visualisation de la répartition des classes
class_counts.plot(kind="bar")
plt.title("Répartition des classes")
plt.ylabel("Nombre d'observations")
plt.xlabel("Classe")
plt.xticks(rotation=0)
plt.show()



## 4. Séparation entre données d'entraînement et données de test

On divise les données en deux parties :

- **train set** : utilisé pour entraîner le modèle ;
- **test set** : utilisé pour évaluer le modèle sur des données non vues pendant l'entraînement.

Ici, on utilise 80 % des données pour l'entraînement et 20 % pour le test.


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Taille X_train :", X_train.shape)
print("Taille X_test  :", X_test.shape)
print("Taille y_train :", y_train.shape)
print("Taille y_test  :", y_test.shape)



## 5. Normalisation des données

k-NN calcule des distances. Si une variable a des valeurs beaucoup plus grandes que les autres, elle peut dominer le calcul de distance.

Même si la base `load_diabetes` est déjà standardisée, nous appliquons ici `StandardScaler` pour montrer la bonne pratique générale.

La normalisation doit être apprise uniquement sur les données d'entraînement, puis appliquée aux données de test.


In [ ]:

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convertir en DataFrame pour affichage
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_train_scaled_df.head()



## 6. Entraînement d'un premier modèle k-NN

On commence avec `k = 5`, une valeur souvent utilisée comme premier essai.


In [ ]:

# Créer le modèle k-NN
knn = KNeighborsClassifier(n_neighbors=5)

# Entraîner le modèle
knn.fit(X_train_scaled, y_train)

# Prédire les classes sur l'ensemble de test
y_pred = knn.predict(X_test_scaled)

# Afficher les premières prédictions
pd.DataFrame({
    "classe_reelle": y_test.values,
    "classe_predite": y_pred
}).head(10)



## 7. Évaluation du modèle

Nous allons utiliser trois outils :

1. **Accuracy** : proportion de prédictions correctes ;
2. **Matrice de confusion** : comparaison entre classes réelles et prédites ;
3. **Rapport de classification** : précision, rappel et F1-score.


In [ ]:

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy du modèle k-NN avec k=5 :", round(accuracy, 4))


In [ ]:

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Risque faible/moyen", "Risque élevé"]
)

disp.plot()
plt.title("Matrice de confusion — k-NN, k=5")
plt.show()


In [ ]:

print(classification_report(
    y_test,
    y_pred,
    target_names=["Risque faible/moyen", "Risque élevé"]
))



## 8. Choix de la meilleure valeur de k

La valeur de `k` influence beaucoup les résultats :

- si `k` est trop petit, le modèle peut être trop sensible au bruit ;
- si `k` est trop grand, le modèle peut devenir trop général et perdre en précision.

Nous allons tester plusieurs valeurs de `k` et comparer l'accuracy.


In [ ]:

k_values = range(1, 31)
accuracies = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    predictions = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, predictions)
    accuracies.append(acc)

results_k = pd.DataFrame({
    "k": list(k_values),
    "accuracy": accuracies
})

results_k.head()


In [ ]:

plt.plot(results_k["k"], results_k["accuracy"], marker="o")
plt.title("Évolution de l'accuracy selon la valeur de k")
plt.xlabel("Valeur de k")
plt.ylabel("Accuracy")
plt.xticks(list(k_values))
plt.grid(True)
plt.show()

best_k = results_k.loc[results_k["accuracy"].idxmax(), "k"]
best_accuracy = results_k["accuracy"].max()

print("Meilleur k :", best_k)
print("Meilleure accuracy :", round(best_accuracy, 4))



## 9. Modèle final avec le meilleur k

On entraîne maintenant un modèle final avec la meilleure valeur de `k` trouvée dans l'étape précédente.


In [ ]:

final_knn = KNeighborsClassifier(n_neighbors=int(best_k))
final_knn.fit(X_train_scaled, y_train)

final_pred = final_knn.predict(X_test_scaled)

print("Accuracy finale :", round(accuracy_score(y_test, final_pred), 4))
print()
print(classification_report(
    y_test,
    final_pred,
    target_names=["Risque faible/moyen", "Risque élevé"]
))



## 10. Comparaison avec un modèle sans normalisation

Dans cette section, on entraîne k-NN sans normaliser les variables. L’objectif est de comprendre l'importance de la normalisation.

Sur certaines bases de données, la différence peut être très importante. Dans la base `load_diabetes`, les variables sont déjà relativement standardisées, donc l'écart peut être faible.


In [ ]:

knn_no_scaling = KNeighborsClassifier(n_neighbors=int(best_k))
knn_no_scaling.fit(X_train, y_train)

pred_no_scaling = knn_no_scaling.predict(X_test)

acc_no_scaling = accuracy_score(y_test, pred_no_scaling)
acc_scaling = accuracy_score(y_test, final_pred)

print("Accuracy avec normalisation    :", round(acc_scaling, 4))
print("Accuracy sans normalisation    :", round(acc_no_scaling, 4))



## 11. Prédire la classe d'un nouveau patient

On peut utiliser le modèle final pour prédire la classe d’un nouveau patient.

Dans cet exemple, nous prenons simplement le premier individu de la base de test.


In [ ]:

# Choisir un individu dans la base de test
new_patient = X_test.iloc[[0]]

# Normaliser avec le même scaler
new_patient_scaled = scaler.transform(new_patient)

# Prédiction
predicted_class = final_knn.predict(new_patient_scaled)[0]
predicted_proba = final_knn.predict_proba(new_patient_scaled)[0]

print("Données du patient :")
display(new_patient)

print("Classe prédite :", predicted_class)
print("Probabilité classe 0 :", round(predicted_proba[0], 4))
print("Probabilité classe 1 :", round(predicted_proba[1], 4))

if predicted_class == 1:
    print("Interprétation : risque élevé de progression.")
else:
    print("Interprétation : risque faible ou moyen de progression.")



## 12. Questions de TP

Répondez aux questions suivantes :

1. Quel est le principe de l’algorithme k-NN ?
2. Pourquoi faut-il normaliser les variables avant d’utiliser k-NN ?
3. Que se passe-t-il lorsque `k = 1` ?
4. Que se passe-t-il lorsque `k` est très grand ?
5. Quelle est la meilleure valeur de `k` trouvée dans ce TP ?
6. L’accuracy suffit-elle pour évaluer un modèle de classification ? Justifiez.
7. Dans la matrice de confusion, quel type d’erreur serait le plus grave dans un contexte médical : faux positif ou faux négatif ? Pourquoi ?



## 13. Exercices supplémentaires

### Exercice 1

Modifier la taille de l'ensemble de test :

```python
test_size=0.30
```

Comparer les résultats avec ceux obtenus avec `test_size=0.20`.

### Exercice 2

Tester les distances suivantes dans `KNeighborsClassifier` :

```python
metric="euclidean"
metric="manhattan"
```

Comparer les résultats.

### Exercice 3

Créer une fonction qui prend une valeur de `k` en entrée et retourne l'accuracy du modèle.

### Exercice 4

Afficher la matrice de confusion du meilleur modèle.

### Exercice 5

Remplacer la transformation de la cible par un autre seuil, par exemple le 75e percentile :

```python
threshold = y_regression.quantile(0.75)
```

Observer l’effet sur la répartition des classes et sur la performance du modèle.



## Conclusion

Dans ce TP, nous avons appliqué l’algorithme **k-NN** à une base de données liée au diabète.

Les points clés à retenir sont :

- k-NN classe un individu selon les classes de ses voisins les plus proches ;
- le choix de `k` est très important ;
- la normalisation est essentielle car l’algorithme repose sur les distances ;
- il faut toujours évaluer un modèle avec plusieurs métriques, pas seulement l’accuracy ;
- dans un problème médical, l’interprétation des erreurs est aussi importante que le score global.
